In [1]:
import jieba
import random
import pkuseg
import zhconv
import pickle
import json
import numpy as np
import sklearn.feature_extraction.text as TfidfVectorizer


In [2]:
class Single_tokenizer():
    def cut(self, words):
        seg_words = list(words)
        return seg_words


In [6]:
class Date_Process():
    def __init__(self, file_path, tokenizer_name, userdict_path, stopwords_path, cleaning_parameters):
        self.file_path = file_path
        self.tokenizer_name = tokenizer_name
        self.userdict_path = userdict_path
        self.stopwords_path = stopwords_path
        self.cleaning_parameters = cleaning_parameters
        self.tokenizer = None
        self.stopwords = []
        self.raw_all_data = []
        self.all_data = []
        self.train_data = []
        self.test_data = []
        self.word2id = {}
        self.id2word = {}
        self.tag2id = {}
        self.id2tag = {}

    def init(self):
        self.read_data()
        self.tokenizer, self.stopwords = self.make_tokenizer(self.tokenizer_name, self.userdict_path, self.stopwords_path)
        self.data_process()

    # 读取数据
    def read_data(self):
        with open(self.file_path, "r", encoding="utf-8") as f:
            for line in f:
                one_line = json.loads(line)
                self.raw_all_data.append([one_line['sentence'], one_line['label_desc'][5:]])

    # 选取分词器
    def make_tokenizer(self, tokenizer_name, userdict_path="", stopwords_path=""):
        if stopwords_path != "":
            # 将停用词读出来放在stopwords这个列表中
            stopwords = [line.strip() for line in open(stopwords_path, 'r', encoding='utf-8').readlines()]
        else:
            stopwords = []
        if tokenizer_name == "pkuseg":
            if userdict_path != "":
                pku = pkuseg.pkuseg(user_dict=userdict_path)
            else:
                pku = pkuseg.pkuseg()
            return pku, stopwords
        elif tokenizer_name == "single":
            single_tokenizer = Single_tokenizer()
            return single_tokenizer, stopwords
        else:
            # 默认使用结巴分词
            if userdict_path != "":
                jieba.load_userdict(userdict_path)
            return jieba, stopwords

    # 全角字符到半角字符的转换
    def full2half(self, string):
        rstring = ""
        for char in string:
            inside_code = ord(char)
            if inside_code == 12288:
                # 全角空格直接转换
                inside_code = 32
                rstring += chr(inside_code)
            elif inside_code >= 65281 and inside_code <= 65374:
                # 全角字符 (除空格) 根据关系转化
                inside_code -= 65248
                rstring += chr(inside_code)
            else:
                rstring += chr(inside_code)
        return rstring

    # 数据清洗
    def data_cleaning(self, words, cleaning_parameters):
        if cleaning_parameters[0]:
            words = zhconv.convert(words, 'zh-cn')
        if cleaning_parameters[1]:
            words = words.lower()
        if cleaning_parameters[2]:
            words = "".join(words.split())
        if cleaning_parameters[3]:
            words = self.full2half(words)
        return words

    # 数据分词，分词前进行数据清洗
    def word_seg(self, all_data, tokenizer, stopwords, cleaning_parameters):
        segmented_all_data = []
        for sentences, tag in all_data:
            sentences = self.data_cleaning(sentences, cleaning_parameters)
            seg_list = tokenizer.cut(sentences)
            seg_list = [i for i in seg_list if i not in stopwords]
            segmented_all_data.append((seg_list, tag))
        return segmented_all_data

    # 制作词到ID的映射，标签到ID的映射
    def make_map_dict(self, segmented_all_data,tokenizer_name):
        all_tag = []
        all_words = []
        all_words.append('<PAD>')
        all_words.append('<UNK>')
        for seg_list, tag in segmented_all_data:
            if tag not in all_tag:
                all_tag.append(tag)
        tag2id = {all_tag[i]: i for i in range(len(all_tag))}
        id2tag = {v: k for k, v in tag2id.items()}
        if tokenizer_name != 'single':
            # 如果不是以字为特征，则引入tfidf的初始化帮助清洗数据
            all_texts = [" ".join(i[0]) for i in segmented_all_data]
            tfidf_vec = TfidfVectorizer(max_features=15000, max_df=0.8, min_df=2)
            tfidf_mat = tfidf_vec.fit_transform(all_texts)
            for word in tfidf_vec.vocabulary_.keys():
                if word not in all_words:
                    all_words.append(word)
        else:
            # 如果以字为特征，则直接生成word2id,id2word
            for seg_list, tag in segmented_all_data:
                for word in seg_list:
                    if word not in all_words:
                        all_words.append(word)
        word2id = {all_words[i]: i for i in range(len(all_words))}
        id2word = {v: k for k, v in word2id.items()}
        return word2id, id2word, tag2id, id2tag

    # 数据处理
    def data_process(self):
        self.all_data = self.word_seg(self.raw_all_data[:50], self.tokenizer, self.stopwords, self.cleaning_parameters)
        self.word2id, self.id2word, self.tag2id, self.id2tag = self.make_map_dict(self.all_data,self.tokenizer_name)

        random.seed(1)
        random.shuffle(self.all_data)
        data_len = len(self.all_data)
        train = int(data_len * 0.8)
        self.train_data = self.all_data[:train]
        self.test_data = self.all_data[train:]

    # 数据解析
    def data_parse(self, data ,word2id,tag2id,max_len):
        parsed_data = []
        for seg_list, tag in data:
            sent_ids = [word2id[word] if word in word2id else word2id['<UNK>'] for word in seg_list]
            tag_id = tag2id[tag]
            sent_len = len(sent_ids)
            if sent_len > max_len:
                sent_ids = sent_ids[:max_len]
            elif sent_len < max_len:
                sent_ids = sent_ids + [word2id['<PAD>']] * (max_len - sent_len)
            parsed_data.append((sent_ids, tag_id))
        return parsed_data


    # 保存数据
    def save_model(self, save_path):
        with open(save_path, 'wb') as f:
            pickle.dump([self.train_data, self.test_data, 
                         self.tag2id, self.id2tag, self.word2id, self.id2word], f)

    # 恢复数据
    def load_model(self, load_path):
        with open(load_path, 'rb') as f:
            self.train_data, self.test_data, \
            self.tag2id, self.id2tag, self.word2id, self.id2word = pickle.load(f)


In [7]:
file_path = "data/dev.jsonl"
tokenizer_name = "jieba"
userdict_path = "./userdict/userdict.txt"
stopwords_path = "./stopwords/stopwords.txt"
cleaning_parameters = [True, True, True, True]

In [8]:
data_p = Date_Process(file_path, tokenizer_name, userdict_path, stopwords_path, cleaning_parameters)

In [9]:
data_p.init()

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\Gkk\AppData\Local\Temp\jieba.cache
Loading model cost 0.314 seconds.
Prefix dict has been built successfully.


TypeError: 'module' object is not callable

In [ ]:
data_p.word2id[:10]


{'<PAD>': 0,
 '<UNK>': 1,
 '江': 2,
 '疏影': 3,
 '甜甜圈': 4,
 '自拍': 5,
 '迷之': 6,
 '角度': 7,
 '竟': 8,
 '这么': 9,
 '好看': 10,
 '美': 11,
 '吸引': 12,
 '一切': 13,
 '事物': 14,
 '以色列': 15,
 '大规模': 16,
 '空袭': 17,
 '开始': 18,
 '!': 19,
 '伊朗': 20,
 '多': 21,
 '个': 22,
 '军事': 23,
 '目标': 24,
 '遭遇': 25,
 '打击': 26,
 '誓言': 27,
 '对等': 28,
 '反击': 29,
 '出栏': 30,
 '一头': 31,
 '猪': 32,
 '亏损': 33,
 '300': 34,
 '元': 35,
 '究竟': 36,
 '谁': 37,
 '能': 38,
 '笑到': 39,
 '最后': 40,
 '以前': 41,
 '很': 42,
 '火': 43,
 '的': 44,
 '巴铁': 45,
 '为何': 46,
 '现在': 47,
 '只字不提': 48,
 '作为': 49,
 '一': 50,
 '名': 51,
 '酒店': 52,
 '从业': 53,
 '人员': 54,
 '你': 55,
 '经历': 56,
 '过': 57,
 '房客': 58,
 '哪些': 59,
 '特别': 60,
 '没有': 61,
 '素质': 62,
 '行为': 63,
 '走': 64,
 '进': 65,
 '荀子': 66,
 '世界': 67,
 '触摸': 68,
 '二千': 69,
 '年前': 70,
 '心灵': 71,
 '温度': 72,
 '图解': 73,
 '全': 74,
 '要素': 75,
 '领域': 76,
 '高': 77,
 '效益': 78,
 '天津': 79,
 '智能': 80,
 '科技': 81,
 '军民': 82,
 '融合': 83,
 '发展': 84,
 '区块链': 85,
 '投资': 86,
 '心得': 87,
 '做到': 88,
 '就': 89,
 '不': 90,
 '会': 91,
 '亏': 92,

In [ ]:
len(data_p.train_data)

40

In [ ]:
save_path = "./model/news_model.pkl"
data_p.save_model(save_path)

In [ ]:
save_path = "./model/news_model.pkl"
data_p = Date_Process(file_path, tokenizer_name, userdict_path, stopwords_path, cleaning_parameters)
data_p.load_model(save_path)